# My way home: snntorch SNN with rate-coded inputs and spiking action outputs

Two independent policies for Sample Factory's `doom_my_way_home`. Turn none/left/right
and move none/forward/backward are decoded separately (six scores, nine combinations).
Native rewards are retained: +1 for the vest and a small per-tic living penalty.
No defend-the-center weights or reward reference numbers are used.

Use a fresh Linux GPU kernel, with `home_models.py`, `home_env.py`, and `home_worker.py`
in this folder. Start in smoke mode. Train/evaluate/replay execute in isolated workers;
this notebook controller does not allocate GPU memory. Full server integration must
pass the smoke test before committing a long run.


## Exact SNN definition

Pixels scaled to [0,1] are converted to deterministic pulse-density rate-coded binary
spikes. This is **rate coding**, not Poisson resampling: over T steps each pixel emits
approximately pixel_value*T spikes (rounding error at most half a spike). Repeatable
encoding keeps PPO's recomputed action probabilities consistent for a given image.
No running mean subtraction is applied to images, since rates must be nonnegative.

Architecture: input spikes ? Conv+LIF ?3 ? Linear+LIF (512) ? Linear+LIF (6 action
neurons) ? fixed scaled spike counts/rates. There is **no learned layer after output
spikes**. Training samples two categorical actions from softmax(scaled firing rates);
evaluation takes the highest count per group. Ties pick the first index, so ties and
silent groups are measured. The training-only critic is an analog 512-to-1 readout.
All synaptic weights and membrane voltages remain floating point on the GPU.

snntorch.Leaky uses beta=.9, threshold=1, initial membrane=.5, subtractive immediate
reset (`reset_delay=False`), triangular surrogate, and no learned beta/threshold.
Every observation resets all input accumulators and LIF state. STACK>1 supplies image
history but does not add recurrence. Temporal-memory/RSNN behavior is not implemented.

Before fresh training, all four hidden-layer gains and the output gain are calibrated
on random gameplay. The initialized weights and calibrated gains are saved and loaded
unchanged into training workers. Output-rate calibration is approximate, not a guarantee
that every action neuron fires; inspect tie/silence diagnostics before interpreting results.
This new input/output architecture is an experiment, not guaranteed to match the earlier
analog-readout SNN's performance.


## 1. Dependencies and paths

Reuse the existing CUDA PyTorch installation. The SNN additionally needs snntorch. If absent, run `%pip install snntorch` in its intended kernel environment and restart the kernel before running these cells. Do not reinstall CUDA PyTorch automatically.

In [ ]:
import os, sys, json, time, subprocess, signal
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
# Controller intentionally does not initialize CUDA. GPU computation is in workers.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "my way home" / "home_worker.py").is_file()), None)
if ROOT is None: raise RuntimeError("Keep this notebook and its three Python helpers in my way home")
WORK_DIR = ROOT / "my way home"
OUTPUT_ROOT = Path(os.environ.get("HOME_DOOM_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()


## 2. Experiment settings

Fresh refuses existing run directories. Resume/evaluate reconstructs the saved configuration and does not recalibrate. Keep the original name/settings when selecting a saved run. Changing SEED creates a new independent model; keep evaluation seeds fixed.

In [ ]:
MODEL = "snn"
SEED = 0
RUN_TAG = "v1"
RUN_MODE = "fresh"       # fresh, resume, evaluate
ENABLE_TRAINING = False # review settings, then explicitly enable
SMOKE_TEST = True
TOTAL_STEPS = 100_000 if SMOKE_TEST else 8_000_000
STACK = 1               # 1, 2 or 4 recent RGB images; match between ANN and SNN
FRAME_SKIP = 4
T = 8
BETA = 0.9
LOGIT_SCALE = 8.0        # SNN only: fixed multiplier on output firing rates
LEARNING_RATE = 1e-4
BATCH_SIZE = 1024
CHECKPOINT_KIND = "latest"  # numerical evaluation: latest or best
EVAL_SEEDS = list(range(10000, 10010 if SMOKE_TEST else 10300))
VIDEO_SEEDS = [20000, 20001, 20002]
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
WORKERS = max(1, min(8, CPU_BUDGET, int(os.environ.get("DOOM_NUM_WORKERS", max(1,CPU_BUDGET-2)))))
EXPERIMENT = f"home_{MODEL}_stack{STACK}_seed{SEED}_{RUN_TAG}" + ("_smoke" if SMOKE_TEST else "")
if MODEL == "snn":
    EXPERIMENT = f"home_snntorch_rate_T{T}_beta{BETA:g}_stack{STACK}_seed{SEED}_{RUN_TAG}" + ("_smoke" if SMOKE_TEST else "")
assert RUN_MODE in {"fresh","resume","evaluate"}
assert CHECKPOINT_KIND in {"latest","best"}
JOB = dict(model=MODEL,experiment=EXPERIMENT,seed=SEED,mode=RUN_MODE,
    output_root=str(OUTPUT_ROOT),total_steps=TOTAL_STEPS,workers=WORKERS,
    frame_skip=FRAME_SKIP,stack=STACK,T=T,beta=BETA,logit_scale=LOGIT_SCALE,
    learning_rate=LEARNING_RATE,batch_size=BATCH_SIZE,checkpoint_kind=CHECKPOINT_KIND,
    eval_seeds=EVAL_SEEDS,video_seeds=VIDEO_SEEDS,
    calibration_frames=64 if SMOKE_TEST else 256,calibration_batch=32 if SMOKE_TEST else 64,
    target_rate=.15,activity_frames=20 if SMOKE_TEST else 500)
print(json.dumps(JOB,indent=2))
print("No training is launched until ENABLE_TRAINING=True")


## 3. Worker launcher

Read the printed log path while training. If CUDA is out of memory, close unused kernels or obtain an available allocated GPU. Existing checkpoints are trusted project files; unrestricted loading is confined to these workers.

In [ ]:
def run_stage(stage):
    if os.name != "posix": raise RuntimeError("Run on the Linux GPU server")
    stage_job = dict(JOB,stage=stage)
    controller_dir = OUTPUT_ROOT / "controller" / EXPERIMENT
    controller_dir.mkdir(parents=True,exist_ok=True)
    stamp = time.time_ns()
    job_path = controller_dir / f"{stage}_{stamp}.json"
    log_path = controller_dir / f"{stage}_{stamp}.log"
    job_path.write_text(json.dumps(stage_job,indent=2))
    with log_path.open("w") as log:
        process = subprocess.Popen([sys.executable,"-u",str(WORK_DIR/"home_worker.py"),str(job_path)],
            cwd=WORK_DIR,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            last_print=0
            while process.poll() is None:
                if time.monotonic()-last_print>=30:
                    print("Running",stage,"log:",log_path,flush=True)
                    last_print=time.monotonic()
                time.sleep(2)
            if process.returncode:
                print(log_path.read_text(errors="replace")[-12000:])
                raise RuntimeError(f"{stage} failed; inspect {log_path}")
        except BaseException:
            try: os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError: pass
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                process.wait()
            raise
    print(log_path.read_text(errors="replace")[-5000:])
    return log_path


## 4. Train (opt-in)

Smoke mode checks execution, not convergence. Full mode requests 8M steps as a starting budget; inspect learning curves rather than assuming this is enough for navigation. Calibration and setup are outside the recorded training time.

In [ ]:
if RUN_MODE == "evaluate":
    print("Evaluation mode; training skipped.")
elif not ENABLE_TRAINING:
    print("Review the job, then set ENABLE_TRAINING=True and rerun settings and this cell.")
else:
    run_stage("train")


## 5. Evaluate the selected checkpoint

This cell intentionally runs only when you execute it. Uses fixed validation episode seeds and the selected latest/best checkpoint. Goal success is detected by the native +1 vest reward (>0.5 within a step); no shaping is added. Time-to-goal is averaged over successful episodes only. Timeout is checked against the actual engine limit, not Gym terminated/truncated alone.

In [ ]:
run_stage("evaluate")
report_path = OUTPUT_ROOT/"results"/EXPERIMENT/f"evaluation_{CHECKPOINT_KIND}"/"summary.json"
report = json.loads(report_path.read_text())
for key in ["checkpoint_kind","env_steps","mean_return","std_return","SE","goal_success_rate",
            "timeout_rate","mean_time_to_goal_successes_only","mean_decisions","latency_mean_ms",
            "output_tie_fraction","output_silent_fraction","firing_rates_input_hidden_output"]:
    print(key,":",report[key])
print("Saved:",report_path)


## 6. Episode plots and SNN diagnostics

Success and timeout bars refer to one trained checkpoint, not uncertainty across independent training seeds. Compare multiple training seeds separately. Firing rates include input and output neurons; zero-event sparsity is 1 minus firing rate.

In [ ]:
episodes=report["episodes"]
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist([r["return"] for r in episodes],bins=min(20,len(episodes)))
axes[0].set(xlabel="Original episode return",ylabel="Episodes")
axes[1].bar(["Goal reached","Timeout"],[report["goal_success_rate"],report["timeout_rate"]])
axes[1].set(ylim=(0,1),ylabel="Episode fraction")
fig.tight_layout();plt.show()
if report["firing_rates_input_hidden_output"] is not None:
    names=["input","conv1","conv2","conv3","fc","policy output"]
    fig,ax=plt.subplots(figsize=(9,4))
    ax.bar(names,report["firing_rates_input_hidden_output"])
    ax.set(ylim=(0,1),ylabel="Firing rate per neuron per internal step")
    fig.tight_layout();plt.show()


## 7. Best-training-checkpoint video replays

Loads the best recorded training-reward checkpoint separately, not necessarily the one evaluated above. Saves three MP4s and matching per-decision action CSVs. ffmpeg with libx264 must be in PATH. A 100k smoke run may not yet have a best checkpoint; missing best causes an explicit error. Demonstration seeds are separate from validation; reserve another untouched set for final reporting.

In [ ]:
run_stage("replay")
replay_dir=OUTPUT_ROOT/"results"/EXPERIMENT/"replay_best"
replay_report=json.loads((replay_dir/"summary.json").read_text())
print("Replay checkpoint:",replay_report["checkpoint"])
for row in replay_report["episodes"]:
    print("Episode seed:",row["seed"],"success:",row["success"],"return:",row["return"])
    display(Video(str(replay_dir/f"seed{row['seed']}.mp4"),embed=True,width=640))


## Reading results and preserving experiments

Success rate is the main navigation metric; return also reflects how quickly the goal
is reached. Zero successes in a short smoke test does not diagnose a broken model.
A single image may be insufficient for robust navigation; test matched STACK=2/4 in
separate experiments before adding cross-frame spiking memory. Do not change stack/T
or decoder scale on a saved model and call it the same controlled evaluation.

Keep config.json, calibration/initial_state.pt, package versions, checkpoint hashes and
raw episode CSVs. Results are ignored by Git. Evaluation/replay reruns overwrite the
same selected-kind report/videos. Keep training budgets, scenario assets, preprocessing,
frame skip and checkpoint-selection rule matched across ANN/SNN comparisons. No measured
energy savings are claimed. These notebooks do not implement ANN-to-SNN conversion.
